# NYC TLC pipeline: evidence walkthrough

This notebook reads the **outputs of `run_pipeline.py`** (it does not redo the pipeline) and shows the evidence for each graded skill:
retrieval completeness, profiling and validation, the workflow model, and reconciliation against TLC's published counts.

Run the pipeline first: `python run_pipeline.py --months 2026-05 2026-06 2026-07`

In [1]:
import json, duckdb, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW, PROC = ROOT/'data'/'raw', ROOT/'data'/'processed'
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
con = duckdb.connect()
MONTHS = sorted(p.name.split('=')[1] for p in PROC.glob('month=*'))
print('months published:', MONTHS)

months published: ['2026-05', '2026-06', '2026-07']


## 1. Retrieval: how we know it is complete
The manifest records, per object, the URL, bytes on disk vs the server's `Content-Length`, SHA-256, rows and status.

In [2]:
man = json.loads((RAW/'retrieval_manifest.json').read_text(encoding='utf-8'))
pd.DataFrame(man['objects'])[['source','mode','status','rows','bytes','expected_bytes','note']]

,source,mode,status,rows,bytes,expected_bytes,note
0,tlc_trip_records,file,downloaded,4090836,69699174,69699174.0,
1,tlc_trip_records,file,cached,3837248,65465637,65465637.0,
2,tlc_trip_records,file,downloaded,3530109,61685033,61685033.0,
3,tlc_zone_lookup,file,downloaded,265,12331,NaN,
4,nyc_open_data:c5iv-bn4s,api,downloaded,1550,322154,NaN,"2 pages, page_size=1000"
5,nyc_open_data:v6kb-cqej,api,downloaded,1,462,NaN,"1 pages, page_size=1000"


## 2. Profiling: schema drift and the null pattern
May 2026 lacks `request_source`; a quarter of rows share one five-column null pattern (incomplete submissions).

In [3]:
import pyarrow.parquet as pq
files = sorted((RAW/'tlc_parquet').glob('*.parquet'))
cols = {f.stem[-7:]: {fl.name for fl in pq.ParquetFile(f).schema_arrow} for f in files}
allc = sorted(set().union(*cols.values()))
drift = pd.DataFrame({m: [c in s for c in allc] for m, s in cols.items()}, index=allc)
drift[~drift.all(axis=1)]

,2026-05,2026-06,2026-07
request_source,False,True,True


In [4]:
P = f"read_parquet('{(RAW/'tlc_parquet'/'*.parquet').as_posix()}', union_by_name=true)"
con.sql(f'''select passenger_count is null pax_null, RatecodeID is null rc_null, store_and_fwd_flag is null flag_null,
              payment_type=0 pay_type_0, congestion_surcharge is null cong_null, count(*) rows_
       from {P} group by all order by rows_ desc''').df()

,pax_null,rc_null,flag_null,pay_type_0,cong_null,rows_
0,False,False,False,False,False,8519595
1,True,True,True,True,True,2938598


## 3. Validation: rule hits and per-vendor quarantine
Rows are never deleted; each carries a `reasons` list. Vendor 7 fails `NONPOSITIVE_DURATION` on 100 % of rows.

In [5]:
reports = {m: json.loads((PROC/f'month={m}'/'validation_report.json').read_text(encoding='utf-8')) for m in MONTHS}
hits = pd.DataFrame({m: r['rule_hits'] for m, r in reports.items()}).fillna(0).astype(int)
hits.loc['raw_rows'] = [reports[m]['raw_rows'] for m in MONTHS]
hits.loc['valid_rows'] = [reports[m]['valid_rows'] for m in MONTHS]
hits.loc['data_yield_pct'] = [reports[m]['data_yield_pct'] for m in MONTHS]
hits

,2026-05,2026-06,2026-07
ZERO_DISTANCE,113031.000,128106.000,128322.000
NONPOSITIVE_DURATION,52063.000,49807.000,42316.000
UNKNOWN_DROPOFF_ZONE,21726.000,23804.000,25096.000
NONPOSITIVE_TOTAL,15545.000,14928.000,15308.000
NEGATIVE_FARE,14231.000,13648.000,14200.000
UNKNOWN_PICKUP_ZONE,6486.000,6569.000,6200.000
DURATION_OVER_MAX,1430.000,1107.000,1002.000
IMPLAUSIBLE_SPEED,1020.000,992.000,1148.000
DISTANCE_OVER_MAX,136.000,149.000,172.000
PICKUP_OUTSIDE_MONTH,14.000,17.000,46.000


In [6]:
pd.concat([pd.DataFrame(r['by_vendor']).assign(month=m) for m, r in reports.items()]).set_index(['month','vendor_id'])

rows  quarantined  quarantine_pct  nonpositive_duration
month   vendor_id                                                            
2026-05 1           805221        19176            2.38                   285
        2          3226222       131667            4.08                    28
        6             7643          179            2.34                     0
        7            51750        51750          100.00                 51750
2026-06 1           613145        17816            2.91                   246
        2          3165903       148321            4.68                    36
        6             8676          141            1.63                     1
        7            49524        49524          100.00                 49524
2026-07 1           595928        31472            5.28                   245
        2          2884655       149078            5.17                    31
        6             7486           71            0.95                     0
        7            42040        42040          100.00                 42040

In [7]:
g = pd.DataFrame([dict(month=m, **c) for m, r in reports.items() for c in r['gates']])
g[[c for c in ['month','check','status','actual','gap_pct','note'] if c in g.columns]]

,month,check,status,actual,gap_pct,note
0,2026-05,row_count_matches_manifest,PASS,4090836.000,NaN,NaN
1,2026-05,data_yield_pct,PASS,95.043,NaN,NaN
2,2026-05,vendor_7_all_zero_duration,WARN,51750.000,NaN,escalate to TLC / vendor: drop-off time not po...
3,2026-05,reconciliation_vs_tlc_published,PASS,NaN,3.67,NaN
4,2026-06,row_count_matches_manifest,PASS,3837248.000,NaN,NaN
5,2026-06,data_yield_pct,PASS,94.376,NaN,NaN
6,2026-06,vendor_7_all_zero_duration,WARN,49524.000,NaN,escalate to TLC / vendor: drop-off time not po...
7,2026-06,reconciliation_vs_tlc_published,PASS,NaN,0.52,NaN
8,2026-07,row_count_matches_manifest,PASS,3530109.000,NaN,NaN
9,2026-07,data_yield_pct,PASS,93.693,NaN,NaN


## 4. Workflow model: one trip, two events, one outcome
`fact_trip` is organised around the lifecycle (pickup → in-trip → drop-off → payment).

In [8]:
fact = f"read_parquet('{(PROC/'month=2026-06'/'fact_trip.parquet').as_posix()}')"
con.sql(f'''select month, vendor_id, request_source, incomplete_submission, pickup_ts, pickup_borough, dropoff_ts, dropoff_borough,
              round(duration_min,1) duration_min, distance_mi, round(speed_mph,1) speed_mph, is_airport_trip, payment_type_id, total_amount
       from {fact} using sample 5 rows''').df()

,month,vendor_id,request_source,incomplete_submission,pickup_ts,pickup_borough,dropoff_ts,dropoff_borough,duration_min,distance_mi,speed_mph,is_airport_trip,payment_type_id,total_amount
0,2026-06,2,HV0003,True,2026-06-16 17:41:00,Manhattan,2026-06-16 18:13:00,Brooklyn,32.0,8.97,16.8,False,0,54.77
1,2026-06,2,None,False,2026-06-06 04:44:23,Manhattan,2026-06-06 04:57:58,Manhattan,13.6,4.41,19.5,False,1,31.50
2,2026-06,2,None,False,2026-06-09 21:59:07,Manhattan,2026-06-09 22:15:38,Manhattan,16.5,2.14,7.8,False,1,25.62
3,2026-06,2,None,False,2026-06-10 15:13:30,Manhattan,2026-06-10 15:31:04,Manhattan,17.6,1.93,6.6,False,1,24.36
4,2026-06,2,None,False,2026-06-17 11:21:36,Manhattan,2026-06-17 11:26:03,Manhattan,4.5,0.50,6.7,False,1,12.66


In [9]:
con.sql(f'''select pickup_borough, count(*) trips, round(median(duration_min),1) med_min, round(quantile_cont(duration_min,0.9),1) p90_min,
              round(median(speed_mph),1) med_mph, round(100*avg(is_airport_trip::int),1) airport_pct, round(100*avg(incomplete_submission::int),1) incomplete_pct
       from {fact} group by 1 order by trips desc''').df()

,pickup_borough,trips,med_min,p90_min,med_mph,airport_pct,incomplete_pct
0,Manhattan,3210394,13.2,28.4,8.6,1.8,24.1
1,Queens,297212,33.3,63.3,18.7,79.7,14.4
2,Brooklyn,99758,22.0,38.6,11.7,1.7,88.4
3,Bronx,13772,19.6,39.4,13.5,0.5,92.4
4,Staten Island,201,23.0,46.6,19.9,2.0,79.1
5,EWR,109,0.2,4.7,21.0,100.0,0.0


## 5. Reconciliation against TLC's published zone counts
June and July agree within 0.7 %. May is +3.7 %, concentrated in outer-borough zones and Vendor 1.

In [10]:
rec = pd.concat([pd.read_csv(PROC/f'month={m}'/'reconciliation_zones.csv') for m in MONTHS])
rec.groupby(['month','borough'])[['raw_pickups','valid_pickups','published_pickups']].sum().assign(
    gap_pct=lambda d: (100*(d.raw_pickups-d.published_pickups)/d.published_pickups).round(2))

raw_pickups  valid_pickups  published_pickups  gap_pct
month   borough                                                              
2026-05 Bronx                35669          34523            14248.0   150.34
        Brooklyn            158842         152098           105574.0    50.46
        EWR                    504             96                0.0      inf
        Manhattan          3537655        3375357          3495888.0     1.19
        Queens              351348         325698           329888.0     6.51
        Staten Island          332            292              305.0     8.85
        Unknown               4734              0                0.0      inf
2026-06 Bronx                15006          13772            15000.0     0.04
        Brooklyn            106869          99758           106836.0     0.03
        EWR                    569            109                0.0      inf
        Manhattan          3385945        3210394          3373903.0     0.36
        Queens              322053         297212           321319.0     0.23
        Staten Island          237            201              237.0     0.00
        Unknown               4175              0                0.0      inf
2026-07 Bronx                15654          14233            15641.0     0.08
        Brooklyn            101806          94138           101784.0     0.02
        EWR                    558             97                0.0      inf
        Manhattan          3090473        2910456          3074888.0     0.51
        Queens              315084         288240           313972.0     0.35
        Staten Island          334            284              334.0     0.00
        Unknown               3828              0                0.0      inf

In [11]:
rec[rec.month=='2026-05'].sort_values('raw_minus_published', ascending=False).head(10)

,month,zone_id,borough,zone_name,raw_pickups,valid_pickups,published_pickups,raw_minus_published,gap_pct
0,2026-05,264,Unknown,NaN,4734,0,NaN,4734,NaN
1,2026-05,76,Brooklyn,East New York,5381,5265,1560.0,3821,244.94
2,2026-05,42,Manhattan,Central Harlem North,15194,14236,11714.0,3480,29.71
3,2026-05,61,Brooklyn,Crown Heights North,7527,7234,4217.0,3310,78.49
4,2026-05,75,Manhattan,East Harlem South,34338,32283,31712.0,2626,8.28
5,2026-05,39,Brooklyn,Canarsie,3625,3559,1178.0,2447,207.72
6,2026-05,74,Manhattan,East Harlem North,18016,16803,15757.0,2259,14.34
7,2026-05,188,Brooklyn,Prospect-Lefferts Gardens,4187,4023,1942.0,2245,115.60
8,2026-05,89,Brooklyn,Flatbush/Ditmas Park,3865,3735,1693.0,2172,128.29
9,2026-05,41,Manhattan,Central Harlem,21083,19519,18943.0,2140,11.30


## 6. Final evidence table

In [12]:
pd.read_csv(PROC/'metrics_monthly.csv').query("borough in ['ALL NYC','Manhattan','Queens','Brooklyn','Bronx']").set_index(['month','borough'])

raw_rows  valid_trips  data_yield_pct  median_duration_min  p90_duration_min  median_speed_mph  airport_trip_share_pct  median_revenue_per_mile  incomplete_submission_pct  \
month   borough                                                                                                                                                                                 
2026-05 ALL NYC     4090836      3888064           95.04                14.63             36.00              8.92                    7.87                    12.01                      22.52   
        Bronx         35669        34523           96.79                33.78             82.32             11.31                    0.28                     4.92                      34.56   
        Brooklyn     158842       152098           95.75                25.12             61.67             10.28                    1.31                     7.06                      56.56   
        Manhattan   3537655      3375357           95.41                13.38             29.27              8.47                    1.79                    12.93                      21.75   
        Queens       351348       325698           92.70                35.08             67.77             17.59                   74.65                     6.07                      13.28   
2026-06 ALL NYC     3837248      3621446           94.38                14.23             32.90              9.10                    8.17                    12.13                      25.37   
        Bronx         15006        13772           91.78                19.60             39.35             13.48                    0.52                     6.52                      92.41   
        Brooklyn     106869        99758           93.35                21.95             38.58             11.70                    1.66                     7.97                      88.44   
        Manhattan   3385945      3210394           94.82                13.23             28.37              8.63                    1.79                    12.91                      24.13   
        Queens       322053       297212           92.29                33.27             63.33             18.75                   79.68                     6.15                      14.40   
2026-07 ALL NYC     3530109      3307448           93.69                14.25             31.83              9.50                    8.49                    11.49                      26.20   
        Bronx         15654        14233           90.92                19.57             39.00             15.08                    0.65                     6.02                      86.93   
        Brooklyn     101806        94138           92.47                22.15             38.37             12.19                    2.14                     7.38                      84.80   
        Manhattan   3090473      2910456           94.18                13.23             27.73              8.96                    1.76                    12.32                      25.16   
        Queens       315084       288240           91.48                30.23             57.75             20.39                   78.82                     5.95                      14.57   

                   tlc_published_pickups  recon_gap_pct  
month   borough                                          
2026-05 ALL NYC                3945903.0           3.67  
        Bronx                    14248.0         150.34  
        Brooklyn                105574.0          50.46  
        Manhattan              3495888.0           1.19  
        Queens                  329888.0           6.51  
2026-06 ALL NYC                3817295.0           0.52  
        Bronx                    15000.0           0.04  
        Brooklyn                106836.0           0.03  
        Manhattan              3373903.0           0.36  
        Queens                  321319.0           0.23  
2026-07 ALL NYC                3506619.0 